In [ ]:
# Quantum Circuit with Single Output Qubit
# 3 source qubits → entanglement → 1 output qubit (only this is measured)

import sys
sys.path.append('..')

from modul.circuit import Circuit
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Input and training matrices
input_matrix = np.eye(3)
training_matrix = np.random.rand(3, 3)

print("Input Matrix (Identity):")
print(input_matrix)
print("\nTraining Matrix:")
print(training_matrix)

def create_single_output_circuit(input_mat, training_mat):
    """Create circuit: 3 source qubits → 1 output qubit"""
    qc = QuantumCircuit(4, 1)  # 4 qubits, 1 measurement
    
    # Source qubits (0,1,2) with IP-TP-H-IP-TP-H-IP-TP
    for qubit in range(3):
        qc.p(float(input_mat[qubit, 0]), qubit)
        qc.p(float(training_mat[qubit, 0]), qubit)
        qc.h(qubit)
        qc.p(float(input_mat[qubit, 1]), qubit)
        qc.p(float(training_mat[qubit, 1]), qubit)
        qc.h(qubit)
        qc.p(float(input_mat[qubit, 2]), qubit)
        qc.p(float(training_mat[qubit, 2]), qubit)
    
    # Entangle all source qubits to output qubit (3)
    qc.cx(0, 3)
    qc.cx(1, 3) 
    qc.cx(2, 3)
    
    # P-H-P-H-P on output qubit
    output_phases = [1.0, 2.0, 3.0]  # Fixed for reproducibility
    qc.p(output_phases[0], 3)
    qc.h(3)
    qc.p(output_phases[1], 3)
    qc.h(3)
    qc.p(output_phases[2], 3)
    
    # Measure ONLY output qubit
    qc.measure(3, 0)
    
    return qc

# Create and measure initial circuit
simulator = AerSimulator()
initial_circuit = create_single_output_circuit(input_matrix, training_matrix)

print("\nCircuit (3 source → 1 output):")
print(initial_circuit.draw(output='text'))

# Initial measurement
compiled = transpile(initial_circuit, simulator)
job = simulator.run(compiled, shots=1000)
counts = job.result().get_counts(compiled)

print("\nInitial measurement (output qubit only):")
for outcome, count in sorted(counts.items()):
    print(f"Output |{outcome}>: {count} counts ({count/1000:.1%})")

# Target: most probable output
target_output = max(counts, key=counts.get)
print(f"\nTarget output: |{target_output}>")

In [ ]:
# Two different input matrices with single output measurement
print("TWO INPUT MATRICES → SINGLE OUTPUT MEASUREMENT")
print("=" * 60)

# Generate different inputs
input_matrix_1 = np.random.rand(3, 3)
input_matrix_2 = np.random.rand(3, 3) 
shared_training = np.random.rand(3, 3)

print("Input Matrix 1:")
print(input_matrix_1)
print("\nInput Matrix 2:") 
print(input_matrix_2)

# Test both inputs until we get different most probable outputs
max_attempts = 50
for attempt in range(max_attempts):
    # Create circuits for both inputs
    circuit_1 = create_single_output_circuit(input_matrix_1, shared_training)
    circuit_2 = create_single_output_circuit(input_matrix_2, shared_training)
    
    # Measure both
    compiled_1 = transpile(circuit_1, simulator)
    compiled_2 = transpile(circuit_2, simulator)
    
    job_1 = simulator.run(compiled_1, shots=1000)
    job_2 = simulator.run(compiled_2, shots=1000)
    
    counts_1 = job_1.result().get_counts(compiled_1)
    counts_2 = job_2.result().get_counts(compiled_2)
    
    target_1 = max(counts_1, key=counts_1.get)
    target_2 = max(counts_2, key=counts_2.get)
    
    if target_1 != target_2:
        print(f"\nSuccess after {attempt + 1} attempts!")
        print(f"Input 1 → Output |{target_1}> ({counts_1[target_1]/1000:.1%})")
        print(f"Input 2 → Output |{target_2}> ({counts_2[target_2]/1000:.1%})")
        break
    
    # Regenerate if same output
    input_matrix_1 = np.random.rand(3, 3)
    input_matrix_2 = np.random.rand(3, 3)
    shared_training = np.random.rand(3, 3)

# Store results
stored_data = {
    'input_1': input_matrix_1,
    'input_2': input_matrix_2, 
    'training': shared_training,
    'target_1': target_1,
    'target_2': target_2,
    'counts_1': counts_1,
    'counts_2': counts_2
}

print("\nBoth inputs produce different output targets!")

In [ ]:
# Train for Input 1's target output
print("TRAINING FOR INPUT 1'S TARGET OUTPUT")
print("=" * 50)

target_1 = stored_data['target_1']
input_1 = stored_data['input_1']
training_start = stored_data['training'].copy()

print(f"Target for Input 1: |{target_1}>")

def objective_input_1(training_flat):
    training_mat = training_flat.reshape(3, 3)
    
    # Create circuit without measurement for analysis
    qc = QuantumCircuit(4)
    for qubit in range(3):
        qc.p(float(input_1[qubit, 0]), qubit)
        qc.p(float(training_mat[qubit, 0]), qubit)
        qc.h(qubit)
        qc.p(float(input_1[qubit, 1]), qubit) 
        qc.p(float(training_mat[qubit, 1]), qubit)
        qc.h(qubit)
        qc.p(float(input_1[qubit, 2]), qubit)
        qc.p(float(training_mat[qubit, 2]), qubit)
    
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)
    
    output_phases = [1.0, 2.0, 3.0]
    qc.p(output_phases[0], 3)
    qc.h(3)
    qc.p(output_phases[1], 3)
    qc.h(3)
    qc.p(output_phases[2], 3)
    
    # Get marginal probability for output qubit
    state = Statevector.from_instruction(qc)
    probs = state.probabilities()
    
    if target_1 == '0':
        target_prob = sum(probs[i] for i in range(0, 16, 2))  # Even indices
    else:
        target_prob = sum(probs[i] for i in range(1, 16, 2))  # Odd indices
    
    return -target_prob

# Optimize
result_1 = minimize(
    objective_input_1,
    training_start.flatten() * 2 * np.pi,
    method='L-BFGS-B',
    bounds=[(0, 2*np.pi) for _ in range(9)]
)

trained_matrix_1 = result_1.x.reshape(3, 3) / (2 * np.pi)

# Test result
test_circuit_1 = create_single_output_circuit(input_1, trained_matrix_1)
compiled_test = transpile(test_circuit_1, simulator)
job_test = simulator.run(compiled_test, shots=1000)
counts_test_1 = job_test.result().get_counts(compiled_test)

print(f"\nAfter training for Input 1:")
for outcome, count in sorted(counts_test_1.items()):
    if outcome == target_1:
        print(f"**|{outcome}>: {count} counts ({count/1000:.1%}) [TARGET]**")
    else:
        print(f"  |{outcome}>: {count} counts ({count/1000:.1%})")

print(f"\nImprovement: {counts_test_1[target_1]/1000:.1%} vs {stored_data['counts_1'][target_1]/1000:.1%}")

In [ ]:
# Continue training for Input 2's target
print("CONTINUING TRAINING FOR INPUT 2'S TARGET OUTPUT")
print("=" * 50)

target_2 = stored_data['target_2']
input_2 = stored_data['input_2']

print(f"Target for Input 2: |{target_2}>")
print(f"Starting from Input 1's trained matrix...")

def objective_input_2(training_flat):
    training_mat = training_flat.reshape(3, 3)
    
    # Create circuit without measurement
    qc = QuantumCircuit(4)
    for qubit in range(3):
        qc.p(float(input_2[qubit, 0]), qubit)
        qc.p(float(training_mat[qubit, 0]), qubit)
        qc.h(qubit)
        qc.p(float(input_2[qubit, 1]), qubit)
        qc.p(float(training_mat[qubit, 1]), qubit)
        qc.h(qubit)
        qc.p(float(input_2[qubit, 2]), qubit)
        qc.p(float(training_mat[qubit, 2]), qubit)
    
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)
    
    output_phases = [1.0, 2.0, 3.0]
    qc.p(output_phases[0], 3)
    qc.h(3)
    qc.p(output_phases[1], 3)
    qc.h(3)
    qc.p(output_phases[2], 3)
    
    # Get marginal probability for output qubit
    state = Statevector.from_instruction(qc)
    probs = state.probabilities()
    
    if target_2 == '0':
        target_prob = sum(probs[i] for i in range(0, 16, 2))
    else:
        target_prob = sum(probs[i] for i in range(1, 16, 2))
    
    return -target_prob

# Start from trained matrix 1
result_2 = minimize(
    objective_input_2,
    trained_matrix_1.flatten() * 2 * np.pi,
    method='L-BFGS-B', 
    bounds=[(0, 2*np.pi) for _ in range(9)]
)

final_trained_matrix = result_2.x.reshape(3, 3) / (2 * np.pi)

print("\nFinal trained matrix:")
print(final_trained_matrix)

# Test both inputs with final matrix
test_1_final = create_single_output_circuit(input_1, final_trained_matrix)
test_2_final = create_single_output_circuit(input_2, final_trained_matrix)

job_1_final = simulator.run(transpile(test_1_final, simulator), shots=1000)
job_2_final = simulator.run(transpile(test_2_final, simulator), shots=1000)

counts_1_final = job_1_final.result().get_counts()
counts_2_final = job_2_final.result().get_counts()

print(f"\nFINAL RESULTS:")
print(f"Input 1 → |{target_1}>: {counts_1_final.get(target_1, 0)/1000:.1%}")
print(f"Input 2 → |{target_2}>: {counts_2_final.get(target_2, 0)/1000:.1%}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Input 1 results
outcomes_1 = list(counts_1_final.keys())
probs_1 = [counts_1_final[k]/1000 for k in outcomes_1]
bars1 = ax1.bar(outcomes_1, probs_1, color='lightblue')
if target_1 in outcomes_1:
    bars1[outcomes_1.index(target_1)].set_color('darkblue')
ax1.set_title(f'Input 1 → Target |{target_1}>')
ax1.set_ylabel('Probability')
for i, p in enumerate(probs_1):
    ax1.text(i, p + 0.02, f'{p:.1%}', ha='center')

# Input 2 results  
outcomes_2 = list(counts_2_final.keys())
probs_2 = [counts_2_final[k]/1000 for k in outcomes_2]
bars2 = ax2.bar(outcomes_2, probs_2, color='lightgreen')
if target_2 in outcomes_2:
    bars2[outcomes_2.index(target_2)].set_color('darkgreen')
ax2.set_title(f'Input 2 → Target |{target_2}>')
ax2.set_ylabel('Probability')
for i, p in enumerate(probs_2):
    ax2.text(i, p + 0.02, f'{p:.1%}', ha='center')

plt.suptitle('Single Output Qubit Results with Final Trained Matrix')
plt.tight_layout()
plt.show()

print("\n3 source qubits compressed to 1 output qubit via entanglement!")